# Manuscript Layout Region Detection — YOLOv8 Training & Proper Multi-Set Evaluation

Self-driving notebook for training and evaluating YOLOv8 on Indic manuscript page layouts.
Classes: `0: header, 1: footer, 2: main_text, 3: side_text, 4: filler`

### Before Running:
1. In the right sidebar:
   - **Data**: Click **Add Input** (or "+") and add your uploaded `dataset` (or `dataset.zip`).
   - **Settings -> Accelerator**: Select **GPU T4 x2** (or GPU P100).
   - **Settings -> Internet**: Toggle **On** (required for `pip install ultralytics` and downloading weights).
2. Click **Run All**.


In [ ]:
# Step 1: Install Ultralytics and dependencies
!pip install -q "ultralytics>=8.0.0" pyyaml pillow


In [ ]:
# Step 2: Hardware Verification
import torch

if not torch.cuda.is_available():
    print("=" * 70)
    print("WARNING: GPU NOT DETECTED!")
    print("Please enable GPU in the Kaggle right sidebar: Settings -> Accelerator -> GPU T4")
    print("Training on CPU will be extremely slow.")
    print("=" * 70)
else:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU active: {device_name} ({vram_gb:.1f} GB VRAM)")


In [ ]:
# Step 3: Dataset Discovery, Preparation & YAML Generation
import os
import shutil
import zipfile
import yaml
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
DATASET_DIR = WORKING_DIR / "dataset"
CLASSES = ["header", "footer", "main_text", "side_text", "filler"]

print("Scanning /kaggle/input contents...")
found_items = list(INPUT_DIR.rglob("*"))
print(f"Total items in /kaggle/input: {len(found_items)}")
for item in INPUT_DIR.iterdir():
    print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")

if not list(INPUT_DIR.iterdir()):
    raise RuntimeError(
        "\n" + "=" * 70 + "\n"
        "ERROR: /kaggle/input is EMPTY!\n"
        "You must add the dataset to the notebook:\n"
        "1. In the right panel, click '+ Add Data' (or 'Add Input')\n"
        "2. Search for your uploaded dataset (or dataset.zip) and click '+' to add it.\n"
        "=" * 70
    )

# 1. Search for data.yaml or any zip file
found_data_yaml = None
found_zip = None

for p in INPUT_DIR.rglob("*"):
    if p.is_file():
        if p.name == "data.yaml" and not found_data_yaml:
            found_data_yaml = p
        elif p.suffix.lower() == ".zip" and not found_zip:
            found_zip = p

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

if found_zip and not found_data_yaml:
    print(f"Unpacking zip archive: {found_zip}")
    with zipfile.ZipFile(found_zip, "r") as z:
        z.extractall(WORKING_DIR)
    extracted = list(WORKING_DIR.rglob("data.yaml"))
    if extracted:
        found_data_yaml = extracted[0]

if found_data_yaml:
    src_dir = found_data_yaml.parent
    print(f"Found dataset directory at: {src_dir}")
    if src_dir.resolve() != DATASET_DIR.resolve():
        print("Copying dataset to /kaggle/working/dataset for write access...")
        for item in src_dir.iterdir():
            dest = DATASET_DIR / item.name
            if item.is_dir():
                shutil.copytree(item, dest, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dest)
else:
    train_dirs = list(INPUT_DIR.rglob("images/train"))
    if train_dirs:
        src_dir = train_dirs[0].parent.parent
        print(f"Found images/train at: {src_dir}")
        for item in src_dir.iterdir():
            dest = DATASET_DIR / item.name
            if item.is_dir():
                shutil.copytree(item, dest, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dest)
    else:
        raise FileNotFoundError(
            "Could not locate manuscript dataset under /kaggle/input! "
            "Please ensure you added dataset or dataset.zip to inputs."
        )

# Prepare real_test labels
real_labels = DATASET_DIR / "real_test" / "labels"
real_draft = DATASET_DIR / "real_test" / "labels_draft"
if real_draft.exists() and not real_labels.exists():
    shutil.copytree(real_draft, real_labels)

# Generate absolute path YAMLs
data_yaml_path = WORKING_DIR / "data_kaggle.yaml"
data_yaml_content = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test_synthetic",
    "names": {i: name for i, name in enumerate(CLASSES)},
}
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

real_yaml_path = WORKING_DIR / "real_test_kaggle.yaml"
real_test_yaml_content = {
    "path": str(DATASET_DIR / "real_test"),
    "train": "images",
    "val": "images",
    "test": "images",
    "names": {i: name for i, name in enumerate(CLASSES)},
}
with open(real_yaml_path, "w") as f:
    yaml.dump(real_test_yaml_content, f, sort_keys=False)

n_train = len(list((DATASET_DIR / "images" / "train").glob("*.jpg"))) if (DATASET_DIR / "images" / "train").exists() else 0
n_val = len(list((DATASET_DIR / "images" / "val").glob("*.jpg"))) if (DATASET_DIR / "images" / "val").exists() else 0
n_test = len(list((DATASET_DIR / "images" / "test_synthetic").glob("*.jpg"))) if (DATASET_DIR / "images" / "test_synthetic").exists() else 0
n_real = len(list((DATASET_DIR / "real_test" / "images").glob("*.jpg"))) if (DATASET_DIR / "real_test" / "images").exists() else 0

print("\n" + "=" * 60)
print("DATASET PREPARATION SUCCESSFUL!")
print(f"  train:          {n_train} images")
print(f"  val:            {n_val} images")
print(f"  test_synthetic: {n_test} images")
print(f"  real_test:      {n_real} images")
print(f"  YAML config:    {data_yaml_path}")
print("=" * 60)


In [ ]:
# Step 4: Train YOLOv8n (Position-Sensitive Document Layout)
from pathlib import Path
import yaml
from ultralytics import YOLO

WORKING_DIR = Path("/kaggle/working")
DATASET_DIR = WORKING_DIR / "dataset"
data_yaml_path = WORKING_DIR / "data_kaggle.yaml"
CLASSES = ["header", "footer", "main_text", "side_text", "filler"]

# Auto-generate YAML if not already generated
if not data_yaml_path.exists():
    with open(data_yaml_path, "w") as f:
        yaml.dump({
            "path": str(DATASET_DIR),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test_synthetic",
            "names": {i: n for i, n in enumerate(CLASSES)},
        }, f, sort_keys=False)

print("Loading pretrained YOLOv8n model...")
model = YOLO("yolov8n.pt")

print(f"Starting training on {data_yaml_path}...")
results = model.train(
    data=str(data_yaml_path),
    epochs=60,
    patience=20,
    imgsz=1024,
    batch=8,
    seed=42,
    project=str(WORKING_DIR / "runs"),
    name="yolov8n_manuscript",
    # Crucial position-sensitive layout settings:
    mosaic=0.0,      # Do not mix quadrants
    fliplr=0.0,      # Indic script flows left-to-right; preserve margins
    flipud=0.0,      # Headers belong at the top, footers at bottom
    degrees=2.5,     # Slight page tilt
    scale=0.15,      # Slight scale variation
    shear=1.0,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
    verbose=True,
)

best_pt = WORKING_DIR / "runs" / "yolov8n_manuscript" / "weights" / "best.pt"
print("\n" + "=" * 60)
print(f"TRAINING COMPLETE! Best weights saved to: {best_pt}")
print("=" * 60)


In [ ]:
# Step 5: Multi-Set Evaluation (Proper per-class mapping via ap_class_index & conf=0.001)
import json
import os
from pathlib import Path
from ultralytics import YOLO

WORKING_DIR = Path("/kaggle/working")
DATASET_DIR = WORKING_DIR / "dataset"
data_yaml_path = WORKING_DIR / "data_kaggle.yaml"
real_yaml_path = WORKING_DIR / "real_test_kaggle.yaml"
best_pt = WORKING_DIR / "runs" / "yolov8n_manuscript" / "weights" / "best.pt"
CLASSES = ["header", "footer", "main_text", "side_text", "filler"]

if not best_pt.exists():
    raise FileNotFoundError(f"Weights not found at {best_pt}! Did training complete?")

print(f"Loading trained model from {best_pt}...")
trained_model = YOLO(str(best_pt))

def support(label_dir):
    cnt = {c: 0 for c in CLASSES}
    if not os.path.exists(label_dir):
        return cnt
    for f in os.listdir(label_dir):
        if f.endswith('.txt'):
            for ln in open(os.path.join(label_dir, f)):
                if ln.strip():
                    cid = int(ln.split()[0])
                    if 0 <= cid < len(CLASSES):
                        cnt[CLASSES[cid]] += 1
    return cnt

sets = [
    ("val", data_yaml_path, "val", DATASET_DIR / "labels" / "val"),
    ("test_synthetic", data_yaml_path, "test", DATASET_DIR / "labels" / "test_synthetic"),
    ("real_test_DRAFT_labels", real_yaml_path, "val", DATASET_DIR / "real_test" / "labels"),
]

metrics_summary = {}

for name, yml, split, ldir in sets:
    print(f"\nEvaluating {name} at conf=0.001, iou=0.6...")
    r = trained_model.val(
        data=str(yml),
        split=split,
        imgsz=1024,
        conf=0.001,   # Crucial: standard mAP requires full PR curve integration without confidence cutoff
        iou=0.6,
        plots=False,
        verbose=False,
    )
    sup = support(str(ldir))
    per = {c: "N/A (no labels)" for c in CLASSES}
    
    # Correct per-class mapping using r.box.ap_class_index:
    # Ultralytics only includes classes present in the ground truth!
    for c, ap, p, rc in zip(r.box.ap_class_index, r.box.ap50, r.box.p, r.box.r):
        class_name = CLASSES[int(c)]
        per[class_name] = {
            "AP50": round(float(ap), 4),
            "P": round(float(p), 4),
            "R": round(float(rc), 4),
        }

    metrics_summary[name] = {
        "mAP50": round(float(r.box.map50), 4),
        "mAP50-95": round(float(r.box.map), 4),
        "support": sup,
        "per_class": per,
    }
    print(f"--- Results for {name} ---")
    print(json.dumps(metrics_summary[name], indent=2))

metrics_out_path = WORKING_DIR / "metrics.json"
with open(metrics_out_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\nSaved metrics to {metrics_out_path}")


In [ ]:
# Step 6: Generate Visual Predictions on Real Test Pages (conf=0.25 for clean visualization)
from pathlib import Path
import shutil
from ultralytics import YOLO

WORKING_DIR = Path("/kaggle/working")
DATASET_DIR = WORKING_DIR / "dataset"
best_pt = WORKING_DIR / "runs" / "yolov8n_manuscript" / "weights" / "best.pt"
real_pred_dir = WORKING_DIR / "real_test_pred"

if real_pred_dir.exists():
    shutil.rmtree(real_pred_dir)

trained_model = YOLO(str(best_pt))
trained_model.predict(
    source=str(DATASET_DIR / "real_test" / "images"),
    conf=0.25,
    imgsz=1024,
    save=True,
    project=str(WORKING_DIR),
    name="real_test_pred",
)

print(f"Visual predictions generated in: {real_pred_dir}")


In [ ]:
# Step 7: Pack All Artifacts into outputs.zip
import zipfile
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
OUTPUT_ZIP = WORKING_DIR / "outputs.zip"
RUN_DIR = WORKING_DIR / "runs" / "yolov8n_manuscript"
best_pt = RUN_DIR / "weights" / "best.pt"
metrics_path = WORKING_DIR / "metrics.json"

print(f"Creating output package {OUTPUT_ZIP}...")

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    if best_pt.exists():
        z.write(best_pt, arcname="best.pt")
        print("  [+] best.pt")

    if metrics_path.exists():
        z.write(metrics_path, arcname="metrics.json")
        print("  [+] metrics.json")

    for plot_file in [
        "results.csv",
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "PR_curve.png",
        "P_curve.png",
        "R_curve.png",
    ]:
        p = RUN_DIR / plot_file
        if p.exists():
            z.write(p, arcname=f"plots/{plot_file}")
            print(f"  [+] plots/{plot_file}")

    pred_dir = WORKING_DIR / "real_test_pred"
    if pred_dir.exists():
        for pred_img in pred_dir.glob("*.jpg"):
            z.write(pred_img, arcname=f"real_test_pred/{pred_img.name}")
            print(f"  [+] real_test_pred/{pred_img.name}")

size_mb = OUTPUT_ZIP.stat().st_size / (1024 * 1024)
print("\n" + "=" * 60)
print(f"SUCCESS! Output package ready: {OUTPUT_ZIP} ({size_mb:.1f} MB)")
print("DOWNLOAD INSTRUCTIONS:")
print("1. In Kaggle right panel under 'Output' (/kaggle/working), find outputs.zip")
print("2. Download it and extract best.pt into your repo: method_yolo/weights/best.pt")
print("=" * 60)
